# Stage 4 DINOv2 + Qwen Train-CV Policy Packfix

Repeat of the train-CV selected Stage 4 policy with shorter artifact folder names and strict package checks.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, traceback
from collections import Counter

import pandas as pd
import numpy as np
from IPython.display import display

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")
RUN_NAME = "stage4_hybrid_dinov2_qwen_traincv_policy_packfix_pad030"
ARCHIVE_PATH = Path("/kaggle/working/stage4_deliverables_hybrid_dinov2_qwen_traincv_policy_packfix_pad030.tar.gz")

DATASET_ROOT_CANDIDATES = [Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"), Path("/kaggle/input/idid-coco-v3")]
DETECTOR_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-detector-assets/idid-detector-assets"),
    Path("/kaggle/input/idid-detector-assets/idid-detector-assets"),
    Path("/kaggle/input/idid-detector-assets"),
]
STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"
LABELS = ["insulator_ok", "defect_flashover", "defect_broken"]
FEATURE_MODEL_ID = "facebook/dinov2-base"
FEATURE_LOGREG_C = 0.03
print("RUN_NAME:", RUN_NAME)


In [ ]:
def sh(args, cwd: Path | None = None, check: bool = True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout: print(p.stdout)
    if p.stderr: print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def load_jsonl(path):
    rows=[]
    with Path(path).open(encoding="utf-8") as f:
        for line in f:
            if line.strip(): rows.append(json.loads(line))
    return rows

def write_jsonl(path, rows):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with Path(path).open("w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False)+"\n")

def pick_existing(candidates):
    for c in candidates:
        p=Path(c)
        if p.exists(): return p
    raise FileNotFoundError("not found among: " + "; ".join(str(x) for x in candidates))

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


In [ ]:
gpu=sh(["nvidia-smi"], check=False)
if gpu.returncode != 0:
    raise RuntimeError("GPU is required")

DATA_ROOT=None
for root in DATASET_ROOT_CANDIDATES:
    if (root/TRAIN_JSONL_REL).exists() and (root/VAL_JSONL_REL).exists(): DATA_ROOT=root; break
if DATA_ROOT is None: raise FileNotFoundError("stage3_regrouped_v2 not found")

DETECTOR_ROOT=None
for root in DETECTOR_ROOT_CANDIDATES:
    if (root/"data/processed/val/images").exists() and (root/"data/processed/val/annotations.json").exists(): DETECTOR_ROOT=root; break
if DETECTOR_ROOT is None: raise FileNotFoundError("detector assets not found")

train_jsonl=DATA_ROOT/TRAIN_JSONL_REL
stage4_gt_jsonl=pick_existing([DETECTOR_ROOT/"analysis/stage4_gt_remapped.jsonl", DATA_ROOT/VAL_JSONL_REL])
images_dir=DETECTOR_ROOT/"data/processed/val/images"
coco_json=DETECTOR_ROOT/"data/processed/val/annotations.json"
print("DATA_ROOT:", DATA_ROOT)
print("DETECTOR_ROOT:", DETECTOR_ROOT)
print("train_jsonl:", train_jsonl)
print("stage4_gt_jsonl:", stage4_gt_jsonl)


In [ ]:
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
sh(["git", "clone", REPO_URL, str(REPO_DIR)])
sh(["python", "-m", "pip", "install", "-q", "transformers==4.57.1", "accelerate", "scikit-learn", "hydra-core", "omegaconf", "pycocotools", "pyyaml", "tabulate"], cwd=REPO_DIR)
print("Repo ready:", REPO_DIR)


In [ ]:
artifact_dir = REPO_DIR / "reports/operation_next/stage4_context_pad030_qwen_artifacts"
detector_predictions = artifact_dir / "detector_predictions.json"
qwen_predictions = artifact_dir / "qwen_predictions_vlm_labels_v1.jsonl"
baseline_metrics_path = artifact_dir / "baseline_stage4_metrics.json"
for p in [detector_predictions, qwen_predictions, baseline_metrics_path]:
    if not p.exists(): raise FileNotFoundError(p)
print("artifacts:", artifact_dir)
print("baseline metrics:", baseline_metrics_path.read_text(encoding="utf-8"))


In [ ]:
# Recreate the context crops from saved detector predictions.
run_dir = REPO_DIR / "outputs/stage4" / RUN_NAME
pred_crops_dir = run_dir / "02_pred_crops"
hybrid_run_dir = run_dir / "03_hybrid_pred" / "stage4_pred_vlm_hybrid_dinov2"
hybrid_eval_dir = run_dir / "04_eval_hybrid"
report_dir = run_dir / "05_report"
for d in [pred_crops_dir, hybrid_run_dir, hybrid_eval_dir, report_dir]: d.mkdir(parents=True, exist_ok=True)

sh([
    "python", "scripts/export_vlm_crops.py",
    "--bbox-source", "pred",
    "--coco-json", str(coco_json),
    "--images-dir", str(images_dir),
    "--predictions-json", str(detector_predictions),
    "--output-dir", str(pred_crops_dir),
    "--split", "val",
    "--padding-ratio", "0.30",
    "--manifest-name", "pred_manifest.jsonl",
    "--summary-name", "pred_manifest_summary.json",
    "--score-threshold", "0.05",
    "--max-detections-per-image", "100",
], cwd=REPO_DIR, check=True)

pred_manifest = pred_crops_dir / "pred_manifest.jsonl"
pred_rows = load_jsonl(pred_manifest)
qwen_rows = load_jsonl(qwen_predictions)
print("pred manifest rows:", len(pred_rows))
print("qwen prediction rows:", len(qwen_rows))
assert {r["record_id"] for r in pred_rows} == {r["record_id"] for r in qwen_rows}


In [ ]:
# Train DINOv2 classifier on clean train crops, select a flashover confidence threshold by train OOF-CV, then predict Stage 4 pred crops.
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModel, AutoImageProcessor

train_rows=[r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in LABELS]
print("train rows:", len(train_rows), Counter(r["coarse_class"] for r in train_rows))

def resolve_train_crop(row):
    p=Path(row["crop_path"])
    if p.is_absolute() and p.exists(): return p
    for c in [train_jsonl.parent/p, train_jsonl.parent.parent/p, DATA_ROOT/p]:
        if c.exists(): return c
    raise FileNotFoundError(row["crop_path"])

def resolve_pred_crop(row):
    p=Path(row["crop_path"])
    if p.is_absolute() and p.exists(): return p
    for c in [pred_manifest.parent/p, pred_crops_dir/p]:
        if c.exists(): return c
    raise FileNotFoundError(row["crop_path"])

device="cuda" if torch.cuda.is_available() else "cpu"
processor=AutoImageProcessor.from_pretrained(FEATURE_MODEL_ID)
model=AutoModel.from_pretrained(FEATURE_MODEL_ID).to(device)
model.eval()

def embed(paths, batch_size=24):
    feats=[]
    with torch.no_grad():
        for start in range(0, len(paths), batch_size):
            batch_paths=paths[start:start+batch_size]
            images=[Image.open(p).convert("RGB") for p in batch_paths]
            inputs=processor(images=images, return_tensors="pt").to(device)
            out=model(**inputs)
            feat=out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state.mean(dim=1)
            feat=feat.float(); feat=feat/feat.norm(dim=-1, keepdim=True)
            feats.append(feat.cpu().numpy())
            print("embedded", min(start+batch_size, len(paths)), "/", len(paths))
    return np.concatenate(feats, axis=0)

X_train=embed([resolve_train_crop(r) for r in train_rows])
y_train=np.array([r["coarse_class"] for r in train_rows])
X_pred=embed([resolve_pred_crop(r) for r in pred_rows])

# OOF calibration: choose a low-confidence flashover fallback threshold on train only.
skf=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_proba=[]; oof_idx=[]; class_order=None
for fold,(tr,va) in enumerate(skf.split(X_train, y_train), start=1):
    clf_fold=LogisticRegression(C=FEATURE_LOGREG_C, class_weight="balanced", max_iter=5000, random_state=42)
    clf_fold.fit(X_train[tr], y_train[tr])
    if class_order is None: class_order=list(clf_fold.classes_)
    assert list(clf_fold.classes_) == class_order
    prob=clf_fold.predict_proba(X_train[va])
    oof_proba.append(prob); oof_idx.append(va)
    print("fold", fold, "done", len(va))
oof_proba=np.vstack(oof_proba)
oof_idx=np.concatenate(oof_idx)
# restore original row order
tmp=np.zeros_like(oof_proba); tmp[oof_idx]=oof_proba; oof_proba=tmp
classes=class_order

def lowconf_flash_second_best(probs, threshold):
    preds=[]
    for prob in probs:
        order=np.argsort(prob)[::-1]
        top=classes[order[0]]; conf=float(prob[order[0]])
        if top == "defect_flashover" and conf < threshold:
            top=classes[order[1]]
        preds.append(top)
    return np.array(preds)

def hard_preds(probs):
    return np.array([classes[i] for i in np.argmax(probs, axis=1)])

threshold_grid=[0.30,0.32,0.33,0.34,0.35,0.36,0.38,0.40,0.45,0.50]
cal_rows=[]
base_pred=hard_preds(oof_proba)
cal_rows.append({"policy":"hard_train_oof","threshold":"", "accuracy":accuracy_score(y_train, base_pred), "macro_f1":f1_score(y_train, base_pred, labels=LABELS, average="macro", zero_division=0)})
for t in threshold_grid:
    p=lowconf_flash_second_best(oof_proba, t)
    cal_rows.append({"policy":"flash_lowconf_second_best", "threshold":t, "accuracy":accuracy_score(y_train, p), "macro_f1":f1_score(y_train, p, labels=LABELS, average="macro", zero_division=0)})
cal_df=pd.DataFrame(cal_rows)
cal_df.to_csv(report_dir/"train_oof_threshold_selection.csv", index=False)
selected=cal_df[cal_df["policy"]=="flash_lowconf_second_best"].sort_values(["macro_f1","accuracy"], ascending=False).iloc[0]
SELECTED_FLASH_CONF_THRESHOLD=float(selected["threshold"])
print("Selected threshold by train OOF:", SELECTED_FLASH_CONF_THRESHOLD)
display(cal_df.sort_values(["macro_f1","accuracy"], ascending=False))

clf=LogisticRegression(C=FEATURE_LOGREG_C, class_weight="balanced", max_iter=5000, random_state=42)
clf.fit(X_train, y_train)
pred_cls=clf.predict(X_pred)
proba=clf.predict_proba(X_pred)
classes=list(clf.classes_)
classifier_csv=report_dir/"dinov2_pred_crop_predictions.csv"
out=[]
for row, pred, probs in zip(pred_rows, pred_cls, proba):
    rec={"record_id":row["record_id"], "pred_coarse_class":str(pred), "confidence":float(np.max(probs))}
    order=np.argsort(probs)[::-1]
    rec["second_best_class"]=str(classes[order[1]])
    rec["second_best_confidence"]=float(probs[order[1]])
    for cls,val in zip(classes, probs): rec[f"prob_{cls}"]=float(val)
    out.append(rec)
pd.DataFrame(out).to_csv(classifier_csv, index=False)
display(pd.DataFrame(out).head())


In [ ]:
# Merge DINO coarse into saved Qwen reporter predictions and evaluate train-CV-selected policies.
qwen_by_id={r["record_id"]:r for r in qwen_rows}
clf_by_id={r["record_id"]:r for r in pd.read_csv(classifier_csv).to_dict("records")}

POLICIES = [
    {"name": "hard_dinov2", "mode": "hard", "threshold": None},
    {"name": f"secondbest_cv{SELECTED_FLASH_CONF_THRESHOLD:.2f}".replace(".", "p"), "mode": "dinov2_second_best", "threshold": SELECTED_FLASH_CONF_THRESHOLD},
    {"name": f"qwen_veto_cv{SELECTED_FLASH_CONF_THRESHOLD:.2f}".replace(".", "p"), "mode": "qwen_ok_veto_flash", "threshold": SELECTED_FLASH_CONF_THRESHOLD},
]

baseline=read_json(baseline_metrics_path)
comparisons=[]
policy_dirs=[]

for policy in POLICIES:
    policy_name = policy["name"]
    policy_run_dir = run_dir / "03_hybrid_pred" / f"stage4_pred_vlm_{policy_name}"
    policy_eval_dir = run_dir / "04_eval_hybrid" / policy_name
    policy_run_dir.mkdir(parents=True, exist_ok=True)
    policy_eval_dir.mkdir(parents=True, exist_ok=True)
    policy_dirs.append((policy_name, policy_run_dir, policy_eval_dir))

    hybrid=[]; decisions=[]
    for rid,qrow in qwen_by_id.items():
        h=dict(qrow)
        crow=clf_by_id[rid]
        old=str(h.get("coarse_class"))
        dinov2_class=str(crow["pred_coarse_class"])
        second_best=str(crow.get("second_best_class", old))
        conf=float(crow["confidence"])
        new=dinov2_class
        reason="hard_dinov2"
        if policy["mode"] == "dinov2_second_best":
            if dinov2_class == "defect_flashover" and conf < float(policy["threshold"]):
                new=second_best
                reason=f"dinov2_flash_lowconf_second_best_cv_{policy['threshold']:.2f}"
        elif policy["mode"] == "qwen_ok_veto_flash":
            if old == "insulator_ok" and dinov2_class == "defect_flashover" and conf < float(policy["threshold"]):
                new=old
                reason=f"qwen_ok_veto_flash_cv_{policy['threshold']:.2f}"
        h["coarse_class"]=new
        h["annotator_notes"]=(str(h.get("annotator_notes") or "") + f" hybrid policy={policy_name}; DINOv2={dinov2_class}; second={second_best}; confidence={conf:.4f}; Qwen={old}; reporter fields retained").strip()
        hybrid.append(h)
        decisions.append({"record_id":rid,"qwen_class":old,"dinov2_class":dinov2_class,"second_best_class":second_best,"hybrid_class":new,"confidence":conf,"policy":policy_name,"reason":reason,"changed":old!=new})

    write_jsonl(policy_run_dir/"predictions_vlm_labels_v1.jsonl", hybrid)
    pd.DataFrame(decisions).to_csv(policy_run_dir/"hybrid_decisions.csv", index=False)
    sh(["python", "scripts/validate_vlm_labels_v1.py", "--input", str(policy_run_dir/"predictions_vlm_labels_v1.jsonl")], cwd=REPO_DIR, check=True)
    sh([
        "python", "scripts/eval_stage4_detector_to_vlm.py",
        "--gt-jsonl", str(stage4_gt_jsonl),
        "--pred-manifest-jsonl", str(pred_manifest),
        "--pred-vlm-run-dir", str(policy_run_dir),
        "--detector-predictions-json", str(detector_predictions),
        "--coco-json", str(coco_json),
        "--match-iou-threshold", "0.5",
        "--good-crop-iou-threshold", "0.7",
        "--output-dir", str(policy_eval_dir),
    ], cwd=REPO_DIR, check=True)
    metrics=read_json(policy_eval_dir/"stage4_metrics.json")
    comparisons.append({
        "policy": policy_name,
        "selected_threshold": SELECTED_FLASH_CONF_THRESHOLD,
        "baseline_pipeline_correct": baseline["counts"]["pipeline_correct_total"],
        "baseline_pipeline_rate": baseline["rates"]["pipeline_correct_rate"],
        "hybrid_pipeline_correct": metrics["counts"]["pipeline_correct_total"],
        "hybrid_pipeline_rate": metrics["rates"]["pipeline_correct_rate"],
        "delta_correct": metrics["counts"]["pipeline_correct_total"]-baseline["counts"]["pipeline_correct_total"],
        "delta_rate": metrics["rates"]["pipeline_correct_rate"]-baseline["rates"]["pipeline_correct_rate"],
        "hybrid_vlm_correct_on_good": metrics["counts"]["vlm_correct_on_good_crop_total"],
    })
comparison_df=pd.DataFrame(comparisons).sort_values(["hybrid_pipeline_correct", "hybrid_pipeline_rate"], ascending=False)
comparison_df.to_csv(report_dir/"stage4_dinov2_traincv_policy_comparison.csv", index=False)
(report_dir/"stage4_dinov2_traincv_policy_comparison.json").write_text(json.dumps(comparisons, indent=2), encoding="utf-8")
print(comparison_df.to_string(index=False))
display(comparison_df)


In [ ]:
# Report and compact archive.
lines=[
    "# Stage 4 DINOv2 + Qwen Train-CV Policy Eval",
    "",
    f"Feature model: `{FEATURE_MODEL_ID}`",
    f"Classifier: LogisticRegression C={FEATURE_LOGREG_C}, class_weight=balanced",
    f"Selected flashover confidence threshold from train OOF: `{SELECTED_FLASH_CONF_THRESHOLD:.2f}`",
    "",
    "## Train OOF Threshold Selection",
    pd.read_csv(report_dir/"train_oof_threshold_selection.csv").sort_values(["macro_f1","accuracy"], ascending=False).to_markdown(index=False),
    "",
    "## Stage 4 Comparison",
    comparison_df.to_markdown(index=False),
]
for policy_name, policy_run_dir, policy_eval_dir in policy_dirs:
    lines.extend([
        "",
        f"## {policy_name}",
        "",
        "### Metrics",
        json.dumps(read_json(policy_eval_dir/"stage4_metrics.json"), indent=2),
        "",
        "### Error buckets",
        json.dumps(read_json(policy_eval_dir/"stage4_error_breakdown.json"), indent=2),
    ])
(report_dir/"stage4_dinov2_traincv_policy_report.md").write_text("\n".join(lines), encoding="utf-8")
print((report_dir/"stage4_dinov2_traincv_policy_report.md").read_text(encoding="utf-8")[:5000])

package_root = REPO_DIR / "outputs/stage4_dinov2_hybrid_traincv_policy_packfix_package"
if package_root.exists(): shutil.rmtree(package_root)
package_root.mkdir(parents=True, exist_ok=True)
for src in [pred_manifest, classifier_csv, report_dir/"train_oof_threshold_selection.csv", report_dir/"stage4_dinov2_traincv_policy_comparison.csv", report_dir/"stage4_dinov2_traincv_policy_comparison.json", report_dir/"stage4_dinov2_traincv_policy_report.md"]:
    if src.exists(): shutil.copy2(src, package_root/src.name)
for policy_name, policy_run_dir, policy_eval_dir in policy_dirs:
    dst_policy = package_root / policy_name
    dst_policy.mkdir(parents=True, exist_ok=True)
    for src in [
        policy_run_dir/"predictions_vlm_labels_v1.jsonl",
        policy_run_dir/"hybrid_decisions.csv",
        policy_eval_dir/"stage4_metrics.json",
        policy_eval_dir/"stage4_error_breakdown.json",
        policy_eval_dir/"stage4_case_table.csv",
        policy_eval_dir/"stage4_summary.md",
    ]:
        assert src.exists(), f"missing package file: {src}"
        shutil.copy2(src, dst_policy/src.name)
if ARCHIVE_PATH.exists(): ARCHIVE_PATH.unlink()
sh(["tar", "-czf", str(ARCHIVE_PATH), "-C", str(package_root.parent), package_root.name], check=True)
print("Archive:", ARCHIVE_PATH)
shutil.rmtree(REPO_DIR, ignore_errors=True)
print("Cleaned work repo:", REPO_DIR)
